# California Housing Prices: Linear, Ridge, and Lasso Regression



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Set visualization style
sns.set_theme(style="whitegrid")

### 1. Data Loading and Exploratory Data Analysis (EDA)

In [ ]:
# Load the dataset
df = pd.read_csv('housing.csv')
print(f"Dataset Shape: {df.shape}\n")
df.head()

In [ ]:
# Check data types and missing values
df.info()

In [ ]:
# Statistical summary of the dataset
df.describe()

In [ ]:
# Missing values count
df.isnull().sum()

In [ ]:
# Visualize the distribution of the target variable
plt.figure(figsize=(8, 5))
sns.histplot(df['median_house_value'], bins=50, kde=True)
plt.title('Distribution of Median House Value')
plt.xlabel('Median House Value')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Visualize housing prices geographically
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df, 
    x='longitude', y='latitude', 
    hue='median_house_value', palette='coolwarm', 
    size='population', sizes=(10, 200), alpha=0.6
)
plt.title('Housing Prices in California by Location')
plt.show()

In [ ]:
# Correlation matrix to understand relationships between numerical features
plt.figure(figsize=(10, 8))
num_df = df.select_dtypes(include=[np.number])
sns.heatmap(num_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.show()

### 2. Feature Engineering
To provide our models with more meaningful signals, we create new composite features based on domain logic.

In [ ]:
# Creating new composite features
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']

df[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].head()

### 3. Preprocessing and Train/Test Split

In [ ]:
# Separate features and target
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['float64', 'int64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Preprocessing pipelines for numerical and categorical data
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Imputes missing total_bedrooms and bedrooms_per_room
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

# Preprocess the data
X_preprocessed = preprocessor.fit_transform(X)

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

### 4. Model Building and Evaluation

In [ ]:
# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=10.0), # alpha increased to 10 for stronger regularization
    'Lasso Regression': Lasso(alpha=10.0, max_iter=10000)
}

# Dictionary to store results
results = []

for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)
    
    # Predict on test set
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    # Append results
    results.append({'Model': name, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R²': r2})

# Convert results to DataFrame for comparison
results_df = pd.DataFrame(results)

### 5. Comparison Table

In [ ]:
# Display the comparison table
results_df.set_index('Model')

### 6. Compare Coefficients

In [ ]:
# Get feature names after preprocessing
cat_encoder = preprocessor.named_transformers_['cat']['onehot']
cat_feature_names = cat_encoder.get_feature_names_out(cat_cols)
feature_names = list(num_cols) + list(cat_feature_names)

# Create a DataFrame to compare coefficients
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Linear': models['Linear Regression'].coef_,
    'Ridge': models['Ridge Regression'].coef_,
    'Lasso': models['Lasso Regression'].coef_
})

# Visualizing the impact of regularization on coefficients
coef_df['Abs_Linear'] = coef_df['Linear'].abs()
top_features = coef_df.sort_values(by='Abs_Linear', ascending=False).head(10)
top_features.drop('Abs_Linear', axis=1)

### Conclusion
